# Axon SFT — Diagnostic Notebook (cell 11 bottleneck probe)

**Goal:** run SFT for one step at the real `batch_size=16` config with CUDA-synced timing **and live memory** probes, so we find out exactly which section of the training step is eating the 90+ GB that triggers the OOM.

What this notebook does, and nothing else:

1. Pulls latest code from GitHub
2. Mounts Drive and links datasets
3. Builds an SFT config with `batch_size=16`, `num_epochs=1`, `profile_first_n_steps=3`
4. Slices the dataset to 48 samples so "1 epoch" = 3 training steps
5. Runs SFT — each section prints **live** so a mid-step OOM still leaves a trace of every section that completed before the crash

Expected output per step:

```
step 0 start: alloc=1.24GB
step 0   tokenizer_fwd           42 ms   alloc= 3.10GB   peak= 3.42GB
step 0   diffusion_fwd          201 ms   alloc=14.50GB   peak=18.20GB
step 0   edges_from_adj           2 ms   alloc=14.51GB   peak=14.52GB
step 0   constraint_fwd         ...
```

If it OOMs, paste everything from the `step 0 start:` line down to the error — the last section logged is the one that blew up.

**Do not edit anything.** Just run top-to-bottom on a GPU runtime (`Runtime > Change runtime type > GPU`).

## 1. Pull latest code + install deps

In [ ]:
# Pull latest from GitHub
import os

if os.path.exists("Axon"):
    %cd Axon
    !git pull origin main
else:
    !git clone https://github.com/tylermark/Axon.git
    %cd Axon

!pip install -q pydantic pymupdf scipy networkx wandb timm 2>/dev/null

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_mem_gb = (getattr(props, "total_memory", None) or getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU:     {torch.cuda.get_device_name(0)} ({gpu_mem_gb:.1f} GB)")

## 2. Mount Drive + link datasets

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/axon"
CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
DATA_DIR = f"{DRIVE_ROOT}/datasets"

import os
for d in [CKPT_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)

# Link archcad400k into the working directory
from pathlib import Path
local_data = Path("datasets")
local_data.mkdir(exist_ok=True)

src = Path(DATA_DIR) / "archcad400k"
target = local_data / "archcad400k"
if src.exists() and not target.exists():
    os.symlink(str(src), str(target))

assert src.exists(), f"archcad400k not found at {src} — upload it first"
print(f"Dataset linked: {target} -> {src}")

## 3. Build SFT config at batch_size=16 (reproduces the OOM)

In [ ]:
import torch
from src.training.sft import SFTTrainer, SFTConfig

# Diagnostic config: match the real notebook's batch so the OOM reproduces.
sft_config = SFTConfig(
    num_epochs=1,
    batch_size=16,                 # match the real notebook — this is the OOM config
    learning_rate=5e-5,
    checkpoint_dir="/tmp/sft_diag",  # throwaway dir
    device="cuda" if torch.cuda.is_available() else "cpu",
    wandb_enabled=False,
    eval_every_n_epochs=999,       # skip eval
    save_every_n_epochs=999,       # skip checkpoint saves
    num_workers=0,
    profile_first_n_steps=3,       # profile first 3 steps only — enough to see the OOM pattern
)

print(f"Batch size: {sft_config.batch_size}")
print(f"Device:     {sft_config.device}")
print(f"Profile:    first {sft_config.profile_first_n_steps} steps")

## 4. Build models + tiny dataset

In [ ]:
import torch
from src.pipeline.config import AxonConfig
from src.tokenizer.cross_attention import Tokenizer
from src.diffusion.reverse import GraphDiffusionModel
from src.constraints.sat_solver import ConstraintSolver
from src.training.sft_data import SFTDatasetAdapter
from src.training.data_engine import ArchCAD400KDataset

# ── Models ──
config = AxonConfig()
tokenizer_model = Tokenizer(config=config.tokenizer)
diffusion_model = GraphDiffusionModel(config=config.diffusion)
constraint_module = ConstraintSolver(config=config.constraints)

param_count = sum(
    sum(p.numel() for p in m.parameters())
    for m in [tokenizer_model, diffusion_model, constraint_module]
)
print(f"Total params: {param_count:,}")

# ── Dataset ──
base_dataset = ArchCAD400KDataset(data_root="datasets/")
full_dataset = SFTDatasetAdapter(base_dataset, max_nodes=512)
print(f"Full dataset: {len(full_dataset)} samples")

# DIAGNOSTIC: 48 samples → 3 steps at batch=16 (matches profile_first_n_steps).
tiny_dataset = torch.utils.data.Subset(full_dataset, range(min(48, len(full_dataset))))
print(f"Tiny slice:   {len(tiny_dataset)} samples  →  {len(tiny_dataset) // sft_config.batch_size} steps/epoch")

## 5. Run SFT with live timing + memory probes

Watch the output. For each step you should see one `start:` line and then one line per section:

```
step 0 start: alloc=1.24GB
step 0   tokenizer_fwd           42 ms   alloc= 3.10GB   peak= 3.42GB
step 0   diffusion_fwd          201 ms   alloc=14.50GB   peak=18.20GB
step 0   ...
```

`alloc` = live PyTorch allocations after the section; `peak` = peak since the previous section boundary. These lines are printed **before** the next section runs, so even if the run OOMs mid-step, the last logged line is the one that tipped it over.

If the run completes all 3 steps cleanly, there's no OOM — the real training should work too. If it crashes, paste everything from the first `step 0 start:` line down to the error and we'll know exactly which section blew up.

In [ ]:
import logging
# Make sure INFO-level logs from SFTTrainer show up in the notebook.
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

sft_trainer = SFTTrainer(
    tokenizer_model=tokenizer_model,
    diffusion_model=diffusion_model,
    constraint_module=constraint_module,
    dataset=tiny_dataset,
    eval_dataset=None,
    config=sft_config,
)

sft_trainer.train()
print("\nDiagnostic run complete.")